In [19]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import os
import re
import json
import hashlib
import random
import platform
import datetime as dt
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
# ============================================================
# 2. Central configuration
# ============================================================

CONFIG = {
    # Project / versioning
    "project_name": "extremism_text_classification_comparison",
    "dataset_version": "extremism_dataset_v1",
    "split_version": "split_v1_stratified_70_15_15_seed30",
    "random_seed": 30,

    # Kaggle input path from your original notebook.
    # Change this if Kaggle shows the dataset at a different path.
    "data_path": "/kaggle/input/datasets/adityasureshgithub/digital-extremism-detection-curated-dataset/dataset.csv",

    # Columns from the original SLP notebook.
    "text_col": "Original_Message",
    "label_col": "Extremism_Label",

    # Label mapping. Positive class = extremist.
    "label_map": {
        "NON_EXTREMIST": 0,
        "NON-EXTREMIST": 0,
        "NON EXTREMIST": 0,
        "NOT_EXTREMIST": 0,
        "NOT EXTREMIST": 0,
        "0": 0,
        0: 0,
        "EXTREMIST": 1,
        "1": 1,
        1: 1,
    },

    # Split ratios. These must sum to 1.0.
    "train_size": 0.70,
    "validation_size": 0.15,
    "test_size": 0.15,

    # Output location in Kaggle.
    "output_dir": "/kaggle/working/research_foundation",

    # Safety setting.
    # If False and split_assignments.csv already exists in output_dir, the notebook stops
    # instead of silently overwriting the canonical split.
    "overwrite_existing_split": False,
}

OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print(json.dumps(CONFIG, indent=2, default=str))


Output directory: /kaggle/working/research_foundation
{
  "project_name": "extremism_text_classification_comparison",
  "dataset_version": "extremism_dataset_v1",
  "split_version": "split_v1_stratified_70_15_15_seed30",
  "random_seed": 30,
  "data_path": "/kaggle/input/datasets/adityasureshgithub/digital-extremism-detection-curated-dataset/dataset.csv",
  "text_col": "Original_Message",
  "label_col": "Extremism_Label",
  "label_map": {
    "NON_EXTREMIST": 0,
    "NON-EXTREMIST": 0,
    "NON EXTREMIST": 0,
    "NOT_EXTREMIST": 0,
    "NOT EXTREMIST": 0,
    "0": 0,
    "0": 0,
    "EXTREMIST": 1,
    "1": 1,
    "1": 1
  },
  "train_size": 0.7,
  "validation_size": 0.15,
  "test_size": 0.15,
  "output_dir": "/kaggle/working/research_foundation",
  "overwrite_existing_split": false
}


In [21]:
# ============================================================
# 3. Reproducibility and helper functions
# ============================================================

def set_seed(seed: int) -> None:
    """Set seeds for Python and NumPy."""
    random.seed(seed)
    np.random.seed(seed)


def sha256_text(value: str, n_chars: int = 16) -> str:
    """Stable short SHA-256 hash for text or file content."""
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:n_chars]


def sha256_file(path: Path, n_chars: int = 16) -> str:
    """Stable short SHA-256 hash for a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()[:n_chars]


def clean_text(text: object) -> str:
    """Minimal text cleanup used only for consistency, not heavy preprocessing."""
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_label(value: object, label_map: dict) -> int:
    """Map raw label values to integer labels 0/1."""
    if pd.isna(value):
        return np.nan

    # Numeric labels are accepted if they are 0/1.
    if isinstance(value, (int, np.integer)):
        return label_map.get(int(value), np.nan)
    if isinstance(value, (float, np.floating)) and value in [0.0, 1.0]:
        return label_map.get(int(value), np.nan)

    key = str(value).strip().upper()
    return label_map.get(key, np.nan)


def require_columns(df: pd.DataFrame, required_cols: list[str]) -> None:
    """Fail loudly if required dataset columns are missing."""
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )


def assert_split_ratios(config: dict) -> None:
    total = config["train_size"] + config["validation_size"] + config["test_size"]
    if not np.isclose(total, 1.0):
        raise ValueError(f"Split ratios must sum to 1.0. Current sum: {total}")


set_seed(CONFIG["random_seed"])
assert_split_ratios(CONFIG)


In [22]:
# ============================================================
# 4. Load raw dataset
# ============================================================

data_path = Path(CONFIG["data_path"])

if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find dataset at: {data_path}\n\n"
        "In Kaggle, check the right-side Data panel and copy the exact CSV path. "
        "Then update CONFIG['data_path']."
    )

raw_df = pd.read_csv(data_path)

print("Raw dataset shape:", raw_df.shape)
print("Raw dataset columns:")
print(list(raw_df.columns))

display(raw_df.head())


Raw dataset shape: (3000, 2)
Raw dataset columns:
['Original_Message', 'Extremism_Label']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Original_Message,Extremism_Label
0,sixth forms should burn to the ground,EXTREMIST
1,whatever should burn benders to the ground,EXTREMIST
2,factories should burn to the ground,EXTREMIST
3,halal factories should burn to the ground,EXTREMIST
4,nhs should burn to the ground,EXTREMIST


In [23]:
# ============================================================
# 5. Validate required columns
# ============================================================

require_columns(raw_df, [CONFIG["text_col"], CONFIG["label_col"]])

print("Required columns found:")
print("Text column:", CONFIG["text_col"])
print("Label column:", CONFIG["label_col"])

print("\nRaw label distribution:")
display(raw_df[CONFIG["label_col"]].value_counts(dropna=False).to_frame("count"))


Required columns found:
Text column: Original_Message
Label column: Extremism_Label

Raw label distribution:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
Extremism_Label,
NON_EXTREMIST,1603
EXTREMIST,1397


In [25]:
# ============================================================
# 6. Create processed dataset
# ============================================================

text_col = CONFIG["text_col"]
label_col = CONFIG["label_col"]

processed = raw_df[[text_col, label_col]].copy()
processed["original_index"] = raw_df.index
processed["raw_label"] = processed[label_col]

# Clean text and map labels.
processed["text"] = processed[text_col].apply(clean_text)
processed["label"] = processed[label_col].apply(lambda x: normalize_label(x, CONFIG["label_map"]))

# Track rows removed for transparency.
initial_n = len(processed)
missing_text_mask = processed["text"].eq("")
missing_label_mask = processed["label"].isna()

removed_summary = {
    "initial_rows": int(initial_n),
    "empty_or_missing_text_rows": int(missing_text_mask.sum()),
    "unmapped_or_missing_label_rows": int(missing_label_mask.sum()),
    "rows_removed_total": int((missing_text_mask | missing_label_mask).sum()),
}

processed = processed.loc[~(missing_text_mask | missing_label_mask)].copy()
processed["label"] = processed["label"].astype(int)

# Stable row IDs after cleaning. These are the IDs every model notebook should use.
processed = processed.reset_index(drop=True)
processed["row_id"] = [f"ex_{i:06d}" for i in range(len(processed))]
processed["text_hash"] = processed["text"].apply(lambda x: sha256_text(x, n_chars=16))
processed["char_len"] = processed["text"].str.len()
processed["word_len"] = processed["text"].str.split().apply(len)

# Final public-facing processed dataset columns.
processed = processed[
    [
        "row_id",
        "original_index",
        "text",
        "label",
        "raw_label",
        "text_hash",
        "char_len",
        "word_len",
    ]
].copy()

print("Rows removed summary:")
print(json.dumps(removed_summary, indent=2))

print("\nProcessed dataset shape:", processed.shape)
display(processed.head())

print("\nProcessed label distribution:")
display(processed["label"].value_counts(dropna=False).sort_index().to_frame("count"))


Rows removed summary:
{
  "initial_rows": 3000,
  "empty_or_missing_text_rows": 1,
  "unmapped_or_missing_label_rows": 0,
  "rows_removed_total": 1
}

Processed dataset shape: (2999, 8)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_id,original_index,text,label,raw_label,text_hash,char_len,word_len
0,ex_000000,0,sixth forms should burn to the ground,1,EXTREMIST,e0cc81a4d5f0f2b3,37,7
1,ex_000001,1,whatever should burn benders to the ground,1,EXTREMIST,906f97390ef91f64,42,7
2,ex_000002,2,factories should burn to the ground,1,EXTREMIST,50fe44bc8cf60d1c,35,6
3,ex_000003,3,halal factories should burn to the ground,1,EXTREMIST,4bedb3fea5ed5fbe,41,7
4,ex_000004,4,nhs should burn to the ground,1,EXTREMIST,f1a8b2543327a925,29,6



Processed label distribution:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
label,
0,1602
1,1397


In [26]:
# ============================================================
# 7. Dataset audit tables
# ============================================================

# Label distribution.
label_distribution = (
    processed["label"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)
label_distribution["proportion"] = label_distribution["count"] / len(processed)
label_distribution["class_name"] = label_distribution["label"].map({0: "non_extremist", 1: "extremist"})

# Text length summary overall and by label.
length_summary_overall = processed[["char_len", "word_len"]].describe().T.reset_index().rename(columns={"index": "length_metric"})
length_summary_by_label = (
    processed
    .groupby("label")[["char_len", "word_len"]]
    .describe()
)

# Duplicate text report.
duplicate_mask = processed.duplicated(subset=["text_hash"], keep=False)
duplicate_text_report = (
    processed.loc[duplicate_mask, ["row_id", "label", "text_hash", "char_len", "word_len", "text"]]
    .sort_values(["text_hash", "row_id"])
    .copy()
)

print("Label distribution:")
display(label_distribution)

print("\nText length summary:")
display(length_summary_overall)

print("\nDuplicate text rows:", len(duplicate_text_report))
if len(duplicate_text_report) > 0:
    display(duplicate_text_report.head(20))
else:
    print("No duplicated text_hash values found.")


Label distribution:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,label,count,proportion,class_name
0,0,1602,0.534178,non_extremist
1,1,1397,0.465822,extremist



Text length summary:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,length_metric,count,mean,std,min,25%,50%,75%,max
0,char_len,2999.0,122.069690,102.279673,5.0,64.0,95.0,146.0,1379.0
1,word_len,2999.0,22.500167,18.771472,1.0,12.0,18.0,28.0,281.0



Duplicate text rows: 0
No duplicated text_hash values found.


In [27]:
# ============================================================
# 8. Create fixed stratified train/validation/test split
# ============================================================

split_path = OUTPUT_DIR / "split_assignments.csv"

if split_path.exists() and not CONFIG["overwrite_existing_split"]:
    raise FileExistsError(
        f"A split file already exists at {split_path}.\n"
        "To avoid accidentally changing the canonical split, this notebook stopped.\n"
        "Set CONFIG['overwrite_existing_split'] = True only if you intentionally want to regenerate it."
    )

# Make sure each class has enough examples for stratified splitting.
class_counts = processed["label"].value_counts().sort_index()
if class_counts.min() < 3:
    raise ValueError(
        "At least one class has fewer than 3 examples. Stratified train/validation/test splitting is unsafe.\n"
        f"Class counts:\n{class_counts}"
    )

train_size = CONFIG["train_size"]
val_size = CONFIG["validation_size"]
test_size = CONFIG["test_size"]
seed = CONFIG["random_seed"]

# First split: train vs temporary holdout.
train_df, temp_df = train_test_split(
    processed,
    train_size=train_size,
    random_state=seed,
    stratify=processed["label"],
    shuffle=True,
)

# Second split: validation vs test inside the holdout.
val_fraction_of_temp = val_size / (val_size + test_size)
val_df, test_df = train_test_split(
    temp_df,
    train_size=val_fraction_of_temp,
    random_state=seed,
    stratify=temp_df["label"],
    shuffle=True,
)

split_assignments = pd.concat(
    [
        train_df[["row_id", "label", "text_hash"]].assign(split="train"),
        val_df[["row_id", "label", "text_hash"]].assign(split="validation"),
        test_df[["row_id", "label", "text_hash"]].assign(split="test"),
    ],
    axis=0,
).sort_values("row_id").reset_index(drop=True)

# Defensive validation.
assert split_assignments["row_id"].is_unique
assert set(split_assignments["row_id"]) == set(processed["row_id"])
assert split_assignments["split"].isin(["train", "validation", "test"]).all()

print("Split counts:")
display(split_assignments["split"].value_counts().to_frame("count"))

print("\nSplit label distribution:")
split_label_distribution = (
    split_assignments
    .groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)
split_sizes = split_assignments.groupby("split").size().rename("split_count").reset_index()
split_label_distribution = split_label_distribution.merge(split_sizes, on="split")
split_label_distribution["proportion_within_split"] = (
    split_label_distribution["count"] / split_label_distribution["split_count"]
)
split_label_distribution["class_name"] = split_label_distribution["label"].map({0: "non_extremist", 1: "extremist"})
display(split_label_distribution)


Split counts:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
split,
train,2099
test,450
validation,450



Split label distribution:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,split,label,count,split_count,proportion_within_split,class_name
0,test,0,240,450,0.533333,non_extremist
1,test,1,210,450,0.466667,extremist
2,train,0,1121,2099,0.534064,non_extremist
3,train,1,978,2099,0.465936,extremist
4,validation,0,241,450,0.535556,non_extremist
5,validation,1,209,450,0.464444,extremist


In [28]:
# ============================================================
# 9. Save processed dataset and split artifacts
# ============================================================

processed_path = OUTPUT_DIR / "processed_dataset.csv"
processed.to_csv(processed_path, index=False)

# Parquet is convenient but optional; CSV is always saved.
parquet_path = OUTPUT_DIR / "processed_dataset.parquet"
try:
    processed.to_parquet(parquet_path, index=False)
    parquet_saved = True
except Exception as e:
    parquet_saved = False
    print("Parquet save skipped:", repr(e))

split_assignments.to_csv(split_path, index=False)
label_distribution.to_csv(OUTPUT_DIR / "label_distribution.csv", index=False)
split_label_distribution.to_csv(OUTPUT_DIR / "split_label_distribution.csv", index=False)
length_summary_overall.to_csv(OUTPUT_DIR / "text_length_summary.csv", index=False)
duplicate_text_report.to_csv(OUTPUT_DIR / "duplicate_text_report.csv", index=False)

with open(OUTPUT_DIR / "rows_removed_summary.json", "w") as f:
    json.dump(removed_summary, f, indent=2)

print("Saved artifacts:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)


Saved artifacts:
- duplicate_text_report.csv
- label_distribution.csv
- processed_dataset.csv
- processed_dataset.parquet
- rows_removed_summary.json
- split_assignments.csv
- split_label_distribution.csv
- text_length_summary.csv


In [29]:
# ============================================================
# 10. Create dataset manifest
# ============================================================

manifest = {
    "project_name": CONFIG["project_name"],
    "dataset_version": CONFIG["dataset_version"],
    "split_version": CONFIG["split_version"],
    "created_at_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
    "random_seed": CONFIG["random_seed"],
    "source_data_path": str(data_path),
    "source_data_file_name": data_path.name,
    "source_data_sha256_16": sha256_file(data_path),
    "text_col_original": CONFIG["text_col"],
    "label_col_original": CONFIG["label_col"],
    "processed_text_col": "text",
    "processed_label_col": "label",
    "positive_class": 1,
    "positive_class_name": "extremist",
    "negative_class": 0,
    "negative_class_name": "non_extremist",
    "raw_rows": int(len(raw_df)),
    "processed_rows": int(len(processed)),
    "rows_removed_summary": removed_summary,
    "num_duplicates_by_text_hash": int(len(duplicate_text_report)),
    "label_counts": {
        str(int(row["label"])): int(row["count"])
        for _, row in label_distribution.iterrows()
    },
    "label_proportions": {
        str(int(row["label"])): float(row["proportion"])
        for _, row in label_distribution.iterrows()
    },
    "split_ratios_requested": {
        "train": CONFIG["train_size"],
        "validation": CONFIG["validation_size"],
        "test": CONFIG["test_size"],
    },
    "split_counts_actual": {
        split: int(count)
        for split, count in split_assignments["split"].value_counts().to_dict().items()
    },
    "artifact_files": {
        "processed_dataset_csv": "processed_dataset.csv",
        "processed_dataset_parquet": "processed_dataset.parquet" if parquet_saved else None,
        "split_assignments": "split_assignments.csv",
        "label_distribution": "label_distribution.csv",
        "split_label_distribution": "split_label_distribution.csv",
        "duplicate_text_report": "duplicate_text_report.csv",
    },
    "software_environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
    },
    "notes": [
        "All downstream model notebooks must reuse this processed dataset and split_assignments.csv.",
        "Hyperparameter tuning must use the train and validation splits only.",
        "The test split must be used only for final locked evaluation.",
    ],
}

manifest_path = OUTPUT_DIR / "dataset_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))


{
  "project_name": "extremism_text_classification_comparison",
  "dataset_version": "extremism_dataset_v1",
  "split_version": "split_v1_stratified_70_15_15_seed30",
  "created_at_utc": "2026-06-20T20:50:21Z",
  "random_seed": 30,
  "source_data_path": "/kaggle/input/datasets/adityasureshgithub/digital-extremism-detection-curated-dataset/dataset.csv",
  "source_data_file_name": "dataset.csv",
  "source_data_sha256_16": "b77520495635b0b9",
  "text_col_original": "Original_Message",
  "label_col_original": "Extremism_Label",
  "processed_text_col": "text",
  "processed_label_col": "label",
  "positive_class": 1,
  "positive_class_name": "extremist",
  "negative_class": 0,
  "negative_class_name": "non_extremist",
  "raw_rows": 3000,
  "processed_rows": 2999,
  "rows_removed_summary": {
    "initial_rows": 3000,
    "empty_or_missing_text_rows": 1,
    "unmapped_or_missing_label_rows": 0,
    "rows_removed_total": 1
  },
  "num_duplicates_by_text_hash": 0,
  "label_counts": {
    "0": 16

/tmp/ipykernel_58/12076941.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
# ============================================================
# 11. Final sanity checks before using this split downstream
# ============================================================

# 1. No missing IDs.
assert processed["row_id"].notna().all()
assert split_assignments["row_id"].notna().all()

# 2. IDs match exactly.
assert set(processed["row_id"]) == set(split_assignments["row_id"])

# 3. No duplicate row IDs.
assert processed["row_id"].is_unique
assert split_assignments["row_id"].is_unique

# 4. Binary labels only.
assert set(processed["label"].unique()).issubset({0, 1})
assert set(split_assignments["label"].unique()).issubset({0, 1})

# 5. All splits present.
assert set(split_assignments["split"].unique()) == {"train", "validation", "test"}

# 6. Split labels match processed labels.
label_check = split_assignments[["row_id", "label"]].merge(
    processed[["row_id", "label"]],
    on="row_id",
    suffixes=("_split", "_processed"),
)
assert (label_check["label_split"] == label_check["label_processed"]).all()

print("All sanity checks passed.")
print("\nThis split is now the canonical split for the full 10-model comparison project.")


All sanity checks passed.

This split is now the canonical split for the full 10-model comparison project.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
# ============================================================
# 12. Create a zip file for download / Google Drive upload
# ============================================================

import shutil

zip_base = Path("/kaggle/working/research_foundation_outputs")
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

print("Created zip:", zip_path)


Created zip: /kaggle/working/research_foundation_outputs.zip
